# Atividade Prática 11 - Lidando com Dados do Mundo Real

Este _notebook_ é responsável por acessar o banco de dados disponibilizado pela MusicBrainz[1] e salvar as informações das músicas
buscadas, essas informações são desnormalizadas e salvas em um arquivo `.csv`. 

## Instalação

Para instalar foi utilizado o Podman[2] para criar uma um container Postgres[3] onde foi instalado a biblioteca Python chamada `mbslave`[4]. 

Os dados foram instalados do repositório da musicbrainz e importados para a instância postgres configurada com base no repositório[4].

Para iniciar o ambiente em container do banco de dados utilize o comando dentro da pasta `docker`:
```
podman-compose up -d
```
Caso necessário crie a pasta `data` dentro da pasta `docker` para que o volume do postgres seja armazenado nela.

Para acessar o banco de dados foi utilizado o pacote `psycopg`[5].

In [1]:
import psycopg2 

def db_connect():
    '''Conecta com o banco de dados para realizar as buscas, retorna a conexão'''
    DB_HOST = '127.0.0.1'
    DB_PORT = '5432'
    DB_DATABASE = 'musicbrainz'
    DB_USER = 'musicbrainz'
    DB_PASSWORD = 'musicbrainz'

    return psycopg2.connect(host=DB_HOST, port=DB_PORT, dbname=DB_DATABASE, user=DB_USER, password=DB_PASSWORD) 
                                # Cria a conexão com o banco de dados.       

conn = db_connect()
cursor = conn.cursor()

cursor.execute("SELECT * FROM artist a WHERE a.name = 'Rick Astley'") # Executa a consulta
artist = cursor.fetchone() # Retorna uma tupla representando uma linha da tabela

print(f"artist = {artist[2]}")

# Importante para não deixar conexões abertas no banco de dados.
cursor.close()
conn.close()

artist = Rick Astley


## Extração de dados

### Testando Busca de Gênero Musical

In [10]:
# Primeiro um teste para encontrar os gêneros musicais.

artista = 'Ariana Grande'
musica = 'hate that i made you love me'

sql = """
    SELECT DISTINCT
    	t.name
    FROM 
    	recording r 
    	INNER JOIN artist_credit ac ON ac.id = r.artist_credit 
    	INNER JOIN recording_tag rt ON rt.recording = r.id 
    	INNER JOIN tag t ON t.id = rt.tag 
    	INNER JOIN genre g ON g.name = t.name
    WHERE 
    	ac.name = %s
    	AND r.name = %s
"""

conn = db_connect()
cursor = conn.cursor()

cursor.execute(sql, (artista, musica))

generos = cursor.fetchall()

print(f'\nGêneros Encontrados: {[g[0] for g in generos]}')

cursor.close()
conn.close()


Gêneros Encontrados: ['pop']


Exemplo Montando os `DataFrames`

In [3]:
import pandas as pd 

exemplo_df = pd.DataFrame([{'artista': artista, 'musica': musica}])

generos_str = ''
first = True
for g in generos:
    if first:
        first = False 
        generos_str = g[0]
    else :
        generos_str += '|' + g[0]

exemplo_df['generos'] = generos_str

exemplo_df

,artista,musica,generos
0,Rick Astley,Never Gonna Give You Up,arena rock|blue-eyed soul|brutal death metal|d...


### Extraindo Informações para Compor novo `DataFrame`

Como a consulta demora serão criado indíces uma vez que o banco importado não os possui. Os indíces foram criados 
utilizando o padrão:

```
CREATE INDEX IF NOT EXISTS <nome_tabela_i_nome_coluna> ON <nome_tabela> (<coluna>);
```

Tratando se o indíce existe ou não para que não de erro ao executar as células novamente.

In [4]:
instrucoes = [ 
    "CREATE INDEX IF NOT EXISTS recording_i_id ON recording (id)",
    "CREATE INDEX IF NOT EXISTS recording_i_artist_credit ON recording (artist_credit)",
    "CREATE INDEX IF NOT EXISTS artist_credit_i_id ON artist_credit (id)",
    "CREATE INDEX IF NOT EXISTS recording_tag_i_recording ON recording_tag (recording)",
    "CREATE INDEX IF NOT EXISTS recording_tag_i_tag ON recording_tag (tag)",
    "CREATE INDEX IF NOT EXISTS tag_i_id ON tag (id)",
    "CREATE INDEX IF NOT EXISTS tag_i_name ON tag (name)",
    "CREATE INDEX IF NOT EXISTS genre_i_name ON genre (name)", 
]

conn = db_connect()
cursor = conn.cursor()

for instrucao in instrucoes:
    print(f'\nExecutando instrução \n{instrucao}')
    cursor.execute(instrucao)

cursor.close()
conn.close()


Executando instrução 
CREATE INDEX IF NOT EXISTS recording_i_id ON recording (id)

Executando instrução 
CREATE INDEX IF NOT EXISTS recording_i_artist_credit ON recording (artist_credit)

Executando instrução 
CREATE INDEX IF NOT EXISTS artist_credit_i_id ON artist_credit (id)

Executando instrução 
CREATE INDEX IF NOT EXISTS recording_tag_i_recording ON recording_tag (recording)

Executando instrução 
CREATE INDEX IF NOT EXISTS recording_tag_i_tag ON recording_tag (tag)

Executando instrução 
CREATE INDEX IF NOT EXISTS tag_i_id ON tag (id)

Executando instrução 
CREATE INDEX IF NOT EXISTS tag_i_name ON tag (name)

Executando instrução 
CREATE INDEX IF NOT EXISTS genre_i_name ON genre (name)


As informações extraídas das músicas serão as tags que indicam os gêneros musica de cada música.

In [5]:


musicas_artistas = pd.read_csv('dados/final/kworb/transformacao/musicas_artistas.csv', sep=';', decimal='.')
                                # Utilizado como base para buscar as informações no banco de dados.
musicas_artistas

,id_musica,musica,id_artista,artista
0,01a02706-e024-7485-80f6-934e371aaaae,For us,01a02706-e015-716c-b555-cc87bebd5c11,&TEAM
1,01a02706-e024-7485-80f6-934f812b0450,Sakura-iro Yell,01a02706-e015-716c-b555-cc87bebd5c11,&TEAM
2,01a02706-e024-7485-80f6-93506f777dc5,We on Fire,01a02706-e015-716c-b555-cc87bebd5c11,&TEAM
3,01a02706-e024-7485-80f6-93516226f055,桜色Yell,01a02706-e015-716c-b555-cc87bebd5c11,&TEAM
4,01a02706-e024-7485-80f6-935203870fca,SonoAide,01a02706-e015-716c-b555-cc88451f7ff6,-真天地開闢集団-ジグザグ
...,...,...,...,...
5051,01a02706-e02d-72a8-bbbd-0643200f6822,Please love yourself(请好好爱自己),01a02706-e01c-70a8-a94d-6981cf95fc3d,金色火种GoldenFire
5052,01a02706-e024-7485-80f6-948994ca0dcb,Recollect,01a02706-e01c-70a8-a94d-69821f2fb23f,鈴木このみ
5053,01a02706-e02d-72a8-bbbd-0644229abe54,Genesis HE★VENS,01a02706-e01c-70a8-a94d-6983ff400ada,鳳 瑛一(CV.緑川 光)、皇 綺羅(CV.小野大輔)、帝 ナギ(CV.代永 翼)、鳳 瑛二...
5054,01a02706-e028-7424-9d5e-11f0a30dc4e4,Voyaging Star's Farewell,01a02706-e01c-70a8-a94d-6984149a3142,鳴潮先約電台


In [ ]:
# Funções 
import traceback

def concatenar_lista(lista, separador=';'):
    lista_str = ''
    first = True
    for item in lista:
        if first:
            first = False 
            lista_str = item
        else :
            lista_str += separador + item
    return lista_str

def concatenar_generos(generos):
    lista = [g[0] for g in generos]

    return concatenar_lista(lista, separador='|')

PATH_ARQUIVO_MUSICA_GENEROS = "dados/musicas_generos.csv"

def escrever_linhas(linhas):
    with open(PATH_ARQUIVO_MUSICA_GENEROS, 'a') as arquivo:
        for linha in linhas:
            arquivo.write(f'{linha}\n')
            
# FIM Funções


# Inicia tratamento do arquivo que será utilizado como armazenamento dos gêneros

with open(PATH_ARQUIVO_MUSICA_GENEROS, 'a') as arquivo:
    pass # Apenas para criar o arquivo sem sobreescrever o que já existe.

arquivo_vazio = False
with open(PATH_ARQUIVO_MUSICA_GENEROS, 'r') as arquivo:
    primeira_linha = arquivo.readline().strip() # Verifica se já existe cabeçalho para o arquivo se não cria. Mas
                                                # caso exista não faz nada para poder executar a célula sem sobreescrever
                                                # o arquivo.
    
    arquivo_vazio = primeira_linha == ''


if arquivo_vazio:
    with open(PATH_ARQUIVO_MUSICA_GENEROS, 'a') as arquivo:
        arquivo.write('id_musica;generos\n') # Escreve o cabeçalho apenas se o arquivo está vazio.


sql = """
    SELECT DISTINCT
    	t.name
    FROM 
    	recording r 
    	INNER JOIN artist_credit ac ON ac.id = r.artist_credit 
    	INNER JOIN recording_tag rt ON rt.recording = r.id 
    	INNER JOIN tag t ON t.id = rt.tag 
    	INNER JOIN genre g ON g.name = t.name
    WHERE 
    	ac.name = %s
    	AND r.name = %s
"""

try: 
    conn = db_connect()
    cursor = conn.cursor()
    
    musicas_generos_df = pd.read_csv(PATH_ARQUIVO_MUSICA_GENEROS, sep=';')
    
    musicas_buscadas = musicas_generos_df['id_musica'].to_list()
    genre_rows = []
    
    print('\nIniciando busca de resultados para:')
    print('checkpoint (5)|id_musica|musica|artista')

    checkpoint = 0
    for id_musica in musicas_artistas['id_musica'].unique():
        if id_musica not in musicas_buscadas: # Condição para não buscar a mesma música me caso de musicas repetidas por 
                                                # conta do número de artistas.
    
            musica_info = musicas_artistas[musicas_artistas['id_musica'] == id_musica]

            musica = musica_info['musica'].iloc[0]
            artista = musica_info['artista'].iloc[0]
        
            print(f'\n{checkpoint}|{id_musica}|{musica}|{artista}|')
        
            cursor.execute(sql, (artista, musica))
            generos = cursor.fetchall()
        
            generos_str = concatenar_generos(generos)
            print(f'Encontrou: {generos_str}')
        
            genre_rows.append(concatenar_lista([id_musica, generos_str]))
            musicas_buscadas.append(id_musica)
            checkpoint += 1 

        if checkpoint == 5: # Salva o arquivo a cada 5 músicas buscadas
            print('\nSalvando Arquivos (checkpoint)...')
            escrever_linhas(genre_rows)
            genre_rows = []
            checkpoint = 0
            
except Exception as e:
    print('\nOcorreu um erro durante a execução')
    print(e)
    traceback.print_exc()

finally: 
    print('\nFechando conexões...')
    cursor.close()
    conn.close()

    print(f'Quantidade de músicas buscadas: {len(genre_rows)}')
    if len(genre_rows) > 0:
        print('Escrevendo gêneros encontrados...')
        escrever_linhas(genre_rows)

    print('Fim')


Iniciando busca de resultados para:
checkpoint (5)|id_musica|musica|artista

0|01a02706-e024-7485-80f6-9492e4426443|Children of Light|Atreyu|
Encontrou: 

1|01a02706-e024-7485-80f6-9493a6483368|Sonic Salvation|August Burns Red|
Encontrou: 

2|01a02706-e024-7485-80f6-949448360e44|On m'invite pas|Aurélie Simon|
Encontrou: 

3|01a02706-e024-7485-80f6-9495a2ed8626|All Night Long|Australian Idol|
Encontrou: 

4|01a02706-e024-7485-80f6-9496aefb8fc2|KiLL iT QUEEN|Ava Max|
Encontrou: 

Salvando Arquivos (checkpoint)...

0|01a02706-e024-7485-80f6-949717208075|Out Of Your Mind|Ava Max|
Encontrou: 

1|01a02706-e024-7485-80f6-94986b1d0f31|The Nights|Avicii|
Encontrou: dance|dancehall|dance-pop|electronic|festival progressive house|house|pop|progressive house|trance

2|01a02706-e024-7485-80f6-949929ea532b|I'm With You|Avril Lavigne|
Encontrou: 

3|01a02706-e024-7485-80f6-949a82065fce|Guillotine Walk|Axel Rudi Pell|
Encontrou: 

4|01a02706-e024-7485-80f6-949bad4ce83c|What You Think of Me|Axel Wilde

### Referências
[1] https://musicbrainz.org (19/08/2026)

[2] https://podman.io (19/08/2026)

[3] https://hub.docker.com/_/postgres (19/08/2026)

[4] https://github.com/acoustid/mbslave (19/08/2026)

[5] https://www.psycopg.org/docs/index.html (20/08/2026)